# Checking results in summary.json files generated by nnUNet

In [ ]:
# imports
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import colormaps
import seaborn as sns
from scipy.stats import ttest_rel
from scipy.stats import wilcoxon
import SimpleITK as sitk

In [ ]:
# path to predicted data
data_path = '../nnUNet_data/nnUNet_raw/Dataset361_Menisci/iwoai_internal_results/'

# list subfolders at this path
subfolders = [f.name for f in os.scandir(data_path) if f.is_dir()]
print(subfolders)

In [ ]:
# what metrics are there?
json_path = os.path.join(data_path, 'clahe_predsTs', 'summary.json')

with open(json_path) as f:
    data = json.load(f)

# extract all metrics
data["metric_per_case"][1]

In [ ]:
# functions to take json file and return array of dice scores, or return hausdorff distance
def get_dice_scores(json_file):
    with open(json_file) as f:
        data = json.load(f)

    # Extract Dice scores from "metric_per_case"
    dice_scores = [
        case["metrics"]["1"]["Dice"] for case in data["metric_per_case"]
    ]

    return dice_scores


def get_hausdorff_distances(json_file):
    with open(json_file) as f:
        data = json.load(f)

    # Extract Hausdorff distances from "metric_per_case"
    hausdorff_distances = [
        case["metrics"]["1"]["Hausdorff_95"] for case in data["metric_per_case"]
    ]

    return hausdorff_distances

In [ ]:
# get only the subfolders that don't contain skmtea
subfolders = [f.name for f in os.scandir(data_path) if f.is_dir() and not f.name.endswith("skmtea")]

# discount images and labels folders (starts with)
subfolders = [f for f in subfolders if not f.startswith("images") and not f.startswith("labels")]

# discount folders that start with ResEnc
subfolders = [f for f in subfolders if not f.startswith("ResEnc")]

# discount hist match
subfolders = [f for f in subfolders if "hist_match" not in f]

# discount fold folders
subfolders = [f for f in subfolders if "fold" not in f]

subfolders

In [ ]:
# reorder the subfolders using the order of the methods (starts with)
method_order = ["zscore_preds", "rescale", "clip_rescale", "hist_eq", "clahe_preds", "nyul", "gmm"]

# Sort by checking which method each subfolder starts with
subfolders = sorted(
    subfolders,
    key=lambda x: next((method_order.index(m) for m in method_order if x.lower().startswith(m.lower())), float('inf'))
)
subfolders

In [ ]:
# for each folder, get dice scores and add to pandas dataframe

# create empty dataframe
internal_df = pd.DataFrame()

for folder in subfolders:
    json_path = os.path.join(data_path, folder, 'summary.json')
    dice_scores = get_dice_scores(json_path)
    internal_df[folder] = dice_scores

column_names = [
    "Z-score",
    "Min-max",
    "R. Min-max",
    "HE",
    "CLAHE",
    "Nyúl",
    "GMM"
]

internal_df.columns = column_names

# turn dice to percentage
internal_df = internal_df * 100
internal_df.describe()

In [ ]:
sns.set_style("whitegrid", {'xtick.bottom': True})

# plot box plot of dice scores
plt.figure(figsize=(10, 6))
sns.boxplot(data=internal_df)
plt.ylabel("DSC", fontsize=18)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.tight_layout()
plt.savefig('boxplot_internal_dice_scores.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# get array of average internal dice scores
internal_dice_means = internal_df.mean()

In [ ]:
# now get amount each method deviates from the mean of methods
mean_of_method_means = internal_dice_means.mean()
internal_dice_deviations = internal_dice_means - mean_of_method_means

# plot bar chart of deviations
plt.figure(figsize=(10, 6))
internal_dice_deviations.plot(kind='bar')
plt.ylabel("Deviation from mean dice score")
plt.show()

In [ ]:
internal_df.corr('pearson')

In [ ]:
# plot pairwise correlation matrix of dice scores, to 3 decimal places
plt.figure(figsize=(10, 6))
sns.heatmap(internal_df.corr('pearson'), annot=True, cmap='coolwarm', fmt=".3f")
plt.title("Pearson correlation matrix of Dice scores")
plt.show()

In [ ]:
# get medians
internal_dice_medians = internal_df.median()

# order columns by decreasing median
internal_dice_medians = internal_dice_medians.sort_values(ascending=False)

In [ ]:
# Perform non-parametric repeated measures ANOVA between all methods 
# using Friedman test
from scipy.stats import friedmanchisquare

# reorder data into correct format
data = internal_df.T.values

# perform friedman test
stat, p = friedmanchisquare(*data)
print(f"Friedman test statistic: {stat}, p-value: {p}")
# if p < 0.05, perform post-hoc tests
if p < 0.05:
    print("Friedman test is significant, can post-hoc tests")

In [ ]:
print("Medians:\n",internal_dice_medians)

In [ ]:
internal_dice_means = internal_dice_means.sort_values(ascending=False)
print("Means:\n",internal_dice_means)

In [ ]:
# Use Bonferroni correction for multiple comparisons
methods = internal_dice_means.index.tolist()

# number of comparisons
n_comparisons = len(methods) * (len(methods) - 1) / 2
alpha = 0.05 / n_comparisons
print(f"Bonferroni corrected alpha: {alpha}")

# for each pair of methods, perform wilcoxon test
p_values = np.zeros((len(methods), len(methods)))

for i, method1 in enumerate(methods):
    for j, method2 in enumerate(methods):
        if i < j:
            t_stat, p_val = wilcoxon(internal_df[method1], internal_df[method2])
            significant = "(SIG)" if p_val < alpha else ""
            if p_val < alpha:
                print(f"{method1} vs {method2}: p-value = {p_val:.7f} {significant}")

            # save p-values to triangular matrix
            p_values[i, j] = p_val

print(p_values)

In [ ]:
df_p_values = pd.DataFrame(p_values, index=methods, columns=methods)
# convert p-values using bonferroni correction
df_p_values_bonf = df_p_values.map(lambda x: min(x * n_comparisons, 1))

# make heatmap of p-values
cmap = colormaps["rocket"]

bad = cmap(0)
low_sig = cmap(0.33)
high_sig= cmap(0.66)
max_sig = cmap(0.99)

# Define custom color mapping
colours = [max_sig, high_sig, low_sig, bad]
# normal significance levels
sig_levels = [0, 0.001, 0.0024, 0.05]
bounds = np.append(np.array(sig_levels), [1]) # Boundaries: 0 → 0.001 → 0.0024 → 0.05 → 1
ticks = sig_levels + [1]

# Create a colormap
cmap = mcolors.LinearSegmentedColormap.from_list("custom_cmap", colours, N=256)
norm = mcolors.BoundaryNorm(bounds, cmap.N)

plt.figure(figsize=(10, 6))
ax = sns.heatmap(df_p_values, annot=True, cmap=cmap, norm=norm, fmt=".3f", mask=df_p_values==0, vmin=0, vmax=1, annot_kws={"size": 14})

# Set custom colorbar ticks and labels
cbar = ax.collections[0].colorbar
cbar.set_ticks(bounds)  # Explicitly include 0.0024
# add '/alpha' to sig_levels
tick_labels = [f"{t}" if t != int(t) else f"{t}" for t in ticks]
cbar.set_ticklabels(tick_labels)  # Force 0.0024 as a label
cbar.ax.tick_params(labelsize=14)

# move x ticks to top
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')

# Adjust tick labels
plt.xticks(fontsize=16, rotation=45)
plt.yticks(fontsize=16)

plt.tight_layout()
plt.savefig('p_values_internal_dice_scores.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
np.array(sig_levels)/n_comparisons

In [ ]:
# Repeat pairwise wilcoxon but with different correction
# use benjamini-hochberg correction for multiple comparisons
from statsmodels.stats.multitest import multipletests

methods = internal_dice_medians.index.tolist()

# get all p-values
p_values = []
comparisons = []
for i, method1 in enumerate(methods):
    for j, method2 in enumerate(methods):
        if i < j:
            t_stat, p_val = wilcoxon(internal_df[method1], internal_df[method2])
            p_values.append(p_val)
            comparisons.append((method1, method2))
# apply benjamini-hochberg correction
corrected_pvals = multipletests(p_values, method='fdr_bh')[1]
# print significant results
for i, (method1, method2) in enumerate(comparisons):
    if corrected_pvals[i] < 0.05:
        print(f"{method1} vs {method2}: p-value = {corrected_pvals[i]:.7f} (BH corrected)")


In [ ]:
!pip install statsmodels

In [ ]:
# plot bland-altman for zscore vs rescale (means and differences)
import statsmodels.api as sm
sm.graphics.mean_diff_plot(internal_df["Zscore"], internal_df["GMM"])
#plt.savefig('bland_altman_zscore_rescale.eps', format='eps', bbox_inches='tight')
plt.show()

In [ ]:
# now get hausdorff distances for each method
internal_hd_df = pd.DataFrame()

for folder in subfolders:
    json_path = os.path.join(data_path, folder, 'summary.json')
    hausdorff_distances = get_hausdorff_distances(json_path)
    internal_hd_df[folder] = hausdorff_distances

internal_hd_df.columns = column_names

internal_hd_df.describe()

In [ ]:
# plot box plot of hausdorff distances
plt.figure(figsize=(10, 6))
sns.boxplot(data=internal_hd_df)
plt.ylabel("Hausdorff distance")
plt.savefig('boxplot_internal_hausdorff_distances.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# check if there is a significant difference between the methods
# Friedman test

# reorder data into correct format
data = internal_hd_df.T.values
# perform friedman test
stat, p = friedmanchisquare(*data)
print(f"Friedman test statistic: {stat}, p-value: {p}")
# if p < 0.05, perform post-hoc tests
if p < 0.05:
    print("Friedman test is significant, can post-hoc tests")

In [ ]:
# check if there is a significant difference between the methods
# perform wilcoxon test between all pairs of methods
# Use Bonferroni correction for multiple comparisons
methods = internal_hd_df.columns.tolist()
# number of comparisons
n_comparisons = len(methods) * (len(methods) - 1) / 2
alpha = 0.05 / n_comparisons

print(f"Bonferroni corrected alpha: {alpha}")
# for each pair of methods, perform wilcoxon test
for i, method1 in enumerate(methods):
    for j, method2 in enumerate(methods):
        if i < j:
            t_stat, p_val = wilcoxon(internal_hd_df[method1], internal_hd_df[method2])
            significant = "(SIG)" if p_val < alpha else ""
            if p_val < 0.1:
                print(f"{method1} vs {method2}: p-value = {p_val:.7f} {significant}")

## External Validation Results

In [ ]:
data_path = '../nnUNet_data/nnUNet_raw/Dataset361_Menisci/skmtea_test_fold_results/'

# get only the subfolders that don't contain skmtea
subfolders = [f.name for f in os.scandir(data_path) if f.is_dir() and f.name.endswith("Ts_skmtea")]

# discount images and labels folders (starts with)
subfolders = [f for f in subfolders if not f.startswith("images") and not f.startswith("labels")]

# discount folders that start with ResEnc
subfolders = [f for f in subfolders if not f.startswith("ResEnc")]

# discount hist match
subfolders = [f for f in subfolders if "hist_match" not in f]

subfolders

In [ ]:
# sort folders
subfolders = sorted(
    subfolders,
    key=lambda x: next((method_order.index(m) for m in method_order if x.lower().startswith(m.lower())), float('inf'))
)

subfolders

In [ ]:
# create empty dataframe
external_df = pd.DataFrame()

for folder in subfolders:
    json_path = os.path.join(data_path, folder, 'summary.json')
    dice_scores = get_dice_scores(json_path)
    external_df[folder] = dice_scores

external_df.columns = column_names

external_df.describe()

In [ ]:
# plot box plot of dice scores
plt.figure(figsize=(10, 6))
sns.boxplot(data=external_df)
plt.ylabel("Dice score")
plt.savefig('boxplot_external_dice_scores.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# remove outlier

In [ ]:
# plot box plot of dice scores
plt.figure(figsize=(10, 6))
sns.boxplot(data=external_df[external_df["Zscore"] > 0.7])
plt.ylabel("Dice score")
plt.savefig('boxplot_external_dice_scores_no_outliers.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Are any of the differences statistically significant?
from scipy.stats import ttest_rel

# perform t-test between all pairs of methods
methods = external_df.columns
for method1 in methods:
    for method2 in methods:
        if method2 == method1:
            continue
        
        t_stat, p_val = ttest_rel(external_df[method1], external_df[method2], alternative="greater")
        significant = "(SIG)" if p_val < 0.05 else ""
        if p_val < 0.05:
            print(f"{method1} vs {method2}: p-value = {p_val:.4f} {significant}")

In [ ]:
from scipy.stats import wilcoxon

# perform wilcoxon signed-rank test between all pairs of methods
methods = external_df.columns
for method1 in methods:
    for method2 in methods:
        if method2 == method1:
            continue
        
        t_stat, p_val = wilcoxon(external_df[method1], external_df[method2], alternative="greater")
        significant = "(SIG)" if p_val < 0.05 else ""
        if p_val < 0.05:
            print(f"{method1} vs {method2}: p-value = {p_val:.4f} {significant}")

In [ ]:
# get hausdorff distances for each method
external_hd_df = pd.DataFrame()

for folder in subfolders:
    json_path = os.path.join(data_path, folder, 'summary.json')
    hausdorff_distances = get_hausdorff_distances(json_path)
    external_hd_df[folder] = hausdorff_distances

external_hd_df.columns = column_names

external_hd_df.describe()

In [ ]:
# plot box plot of hausdorff distances
plt.figure(figsize=(10, 6))
sns.boxplot(data=external_hd_df)
plt.ylabel("Hausdorff distance")
plt.savefig('boxplot_external_hausdorff_distances.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

## Check outlier cases

In [ ]:
external_df.head()

In [ ]:
# external image 3 has a very low dice score for all methods
# let's look at this image
# skmtea paths
im_num = 2 # indexing starts at 0
skmtea_im_path = os.path.join(data_path, 'imagesTs_skmtea')
image_paths = [f.path for f in os.scandir(skmtea_im_path) if f.is_file()]
image_paths.sort()
bad_image_path = image_paths[im_num]

# load image (.nii.gz)
image = sitk.ReadImage(bad_image_path)
image_array = sitk.GetArrayFromImage(image)
print(image_array.shape)

# plot image
plt.figure(figsize=(10, 6))
plt.imshow(image_array[100,...], cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
# load mask
skmtea_mask_path = os.path.join(data_path, 'labelsTs_skmtea')
mask_paths = [f.path for f in os.scandir(skmtea_mask_path) if f.is_file()]
mask_paths.sort()
mask_path = mask_paths[im_num]
mask = sitk.ReadImage(mask_path)
mask_array = sitk.GetArrayFromImage(mask)
print(mask_array.shape)

In [ ]:
# load prediction
zscore_preds_path = os.path.join(data_path, 'zscore_predsTs_skmtea')
pred_paths = [f.path for f in os.scandir(zscore_preds_path) if f.is_file()]
pred_paths.sort()
pred_path = pred_paths[im_num]
pred = sitk.ReadImage(pred_path)
pred_array = sitk.GetArrayFromImage(pred)
print(pred_array.shape)

In [ ]:
# plot both mask and prediction, summing through axis 1
plt.figure(figsize=(18, 10))
plt.subplot(1, 2, 1)
plt.imshow(np.sum(mask_array, axis=1), cmap="gray")
plt.title("Mask")
plt.subplot(1, 2, 2)
plt.imshow(np.sum(pred_array, axis=1), cmap="gray")
plt.title("Prediction")
plt.show()

In [ ]:
# look at slice 115-125, slice by slice (image on left and mask on right)
for i in range(115, 125):
    plt.figure(figsize=(18, 10))
    plt.subplot(1, 3, 1)
    plt.imshow(image_array[i,...], cmap="gray")
    plt.title("Image")
    plt.axis("off")
    plt.subplot(1, 3, 2)
    plt.imshow(mask_array[i,...], cmap="gray")
    plt.title("Mask")
    plt.axis("off")
    plt.subplot(1, 3, 3)
    plt.imshow(pred_array[i,...], cmap="gray")
    plt.title("Prediction")
    plt.axis("off")
    plt.show()

In [ ]:
# look at slice 170 in the z direction on image mask and prediction, rotated 90 degrees
plt.figure(figsize=(18, 10))
plt.subplot(1, 3, 1)
plt.imshow(np.rot90(image_array[...,170]), cmap="gray", aspect = (0.31/0.8))
plt.title("Image")
plt.axis("off")
plt.subplot(1, 3, 2)
plt.imshow(np.rot90(mask_array[...,170]), cmap="gray", aspect = (0.31/0.8))
plt.title("Mask")
plt.axis("off")
plt.subplot(1, 3, 3)
plt.imshow(np.rot90(pred_array[...,170]), cmap="gray", aspect = (0.31/0.8))
plt.title("Prediction")
plt.axis("off")
plt.show()

## Do External Validation on whole of SKMTEA

In [ ]:
data_path = '../nnUNet_data/nnUNet_raw/Dataset361_Menisci/skmtea_external_results/'

# get only the subfolders that end in _all_skmtea
subfolders = [f.name for f in os.scandir(data_path) if f.is_dir() and f.name.endswith("_all_skmtea")]

# discount images and labels folders (starts with)
subfolders = [f for f in subfolders if not f.startswith("images") and not f.startswith("labels") 
              and not f.startswith("ResEnc") and not f.startswith("clahe_hist_match")
              and not f.startswith("zscore_postproc") and "hist_match" not in f and "fold" not in f]
subfolders

In [ ]:
# reorder the subfolders using the order of the methods (starts with)
method_order = ["zscore_preds", "rescale", "clip_rescale", "hist_eq", "clahe_preds", "resenc", "nyul", "gmm"]
# Sort by checking which method each subfolder starts with
subfolders = sorted(
    subfolders,
    key=lambda x: next((method_order.index(m) for m in method_order if x.lower().startswith(m.lower())), float('inf'))
)
subfolders

In [ ]:
# for each folder, get dice scores and add to pandas dataframe

# create empty dataframe
external_all_df = pd.DataFrame()

for folder in subfolders:
    json_path = os.path.join(data_path, folder, 'summary.json')
    dice_scores = get_dice_scores(json_path)
    external_all_df[folder] = dice_scores

# turn dice to percentage
external_all_df = external_all_df * 100
external_all_df.head()

In [ ]:
# rename columns
external_all_df.columns = [
    "Z-score",
    "Min-max",
    "R. Min-max",
    "HE",
    "CLAHE",
    "Nyúl",
    "GMM"
]

In [ ]:
# plot box plot of dice scores
plt.figure(figsize=(10, 6))
sns.boxplot(data=external_all_df)
plt.ylabel("DSC", fontsize=18)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.tight_layout()
plt.savefig('boxplot_external_all_dice_scores.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# now box plots of both internal and external
# as subplots with same y
sns.set_theme(palette="colorblind")
sns.set_style("whitegrid", {'xtick.bottom': True})
plt.subplots(1, 2, figsize=(9, 7), sharey=True)
plt.subplot(1, 2, 1)
sns.boxplot(data=internal_df, showfliers=False)
plt.ylabel("DSC", fontsize=18)
plt.xticks(fontsize=16, rotation=60)
plt.yticks(fontsize=16)
plt.subplot(1, 2, 2)
sns.boxplot(data=external_all_df, showfliers=False)
plt.ylabel("DSC", fontsize=18)
plt.xticks(fontsize=16, rotation=60)
plt.yticks(fontsize=16)

plt.tight_layout()
plt.savefig('boxplot_internal_external_dice_scores.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# now have both internal and external on same plot, grouped by method
# e.g. for zscore, internal and external are next to each other and a different color
# create new dataframe with internal and external data
internal_df["Type"] = "Internal"
external_all_df["Type"] = "External"
# combine dataframes
combined_df = pd.concat([internal_df, external_all_df], axis=0)
# melt dataframe to long format
combined_df = combined_df.melt(id_vars=["Type"], var_name="Method", value_name="DSC")

# Define different flier properties for each type
flierprops_internal = dict(marker='o', color='black', alpha=0.6, markersize=6)  # Circles for Internal
flierprops_external = dict(marker='x', color='red', alpha=0.8, markersize=6)  # X markers for External

# plot box plot of combined data
plt.figure(figsize=(10, 7))
sns.boxplot(data=combined_df, x="Method", y="DSC", hue="Type", showfliers=False, palette='muted', linewidth=1.5)

plt.ylabel("DSC (%)", fontsize=18)
# x label not needed
plt.xlabel("")
plt.xticks(fontsize=16, rotation=30)
plt.yticks(fontsize=16)
# no space so put legend outside upper right
plt.legend(fontsize=16, loc='upper right', bbox_to_anchor=(1.01, 1.18))
plt.tight_layout()
plt.savefig('boxplot_internal_external_dice_scores_grouped.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# remove type column from external_all_df
external_all_df = external_all_df.drop(columns=["Type"])
external_all_df.mean()

In [ ]:
# mean of method dice scores
external_all_dice_means = external_all_df.mean()

# now get amount each method deviates from the mean of methods
mean_of_method_means = external_all_dice_means.mean()
external_all_dice_deviations = external_all_dice_means - mean_of_method_means

# plot bar chart of deviations
plt.figure(figsize=(10, 6))
external_all_dice_deviations.plot(kind='bar')
plt.ylabel("Deviation from mean dice score")

In [ ]:
internal_dice_deviations

In [ ]:
# plot the difference between internal and external mean deviation
deviation_difference = external_all_dice_deviations - internal_dice_deviations
plt.figure(figsize=(10, 6))
deviation_difference.plot(kind='bar')
plt.ylabel("Deviation difference from internal to external dice score")

In [ ]:
# now just plot the difference between internal and external dice score means
mean_difference = external_all_dice_means - internal_dice_means
plt.figure(figsize=(10, 6))
mean_difference.plot(kind='bar')
plt.ylabel("Mean difference between internal and external dice scores")

# axis range
plt.ylim(-0.12, -0.08)

In [ ]:
mean_difference

In [ ]:
# plot pairwise correlation matrix of dice scores, to 3 decimal places
plt.figure(figsize=(10, 6))
sns.heatmap(external_all_df.corr('pearson'), annot=True, fmt=".3f")
plt.title("Pearson correlation matrix of Dice scores")
plt.show()

In [ ]:
# get medians
external_all_dice_medians = external_all_df.median()
# order columns by decreasing median
external_all_dice_medians = external_all_dice_medians.sort_values(ascending=False)

In [ ]:
external_all_dice_medians

In [ ]:
external_all_dice_means = external_all_dice_means.sort_values(ascending=False)
external_all_dice_means

In [ ]:
# Perform non-parametric repeated measures ANOVA between all methods
# using Friedman test
from scipy.stats import friedmanchisquare

# reorder data into correct format
data = external_all_df.T.values

# perform friedman test
stat, p = friedmanchisquare(*data)
print(f"Friedman test statistic: {stat}, p-value: {p}")
# if p < 0.05, perform post-hoc tests
if p < 0.05:
    print("Friedman test is significant, can post-hoc tests")

In [ ]:
# print medians
print("Medians:\n",external_all_dice_medians)

In [ ]:
# Use Bonferroni correction for multiple comparisons
methods = external_all_dice_means.index.tolist()
n_comparisons = len(methods) * (len(methods) - 1) / 2
alpha = 0.05 / n_comparisons

print(f"Bonferroni corrected alpha: {alpha}\n")

p_values_ext = np.zeros((len(methods), len(methods)))

# for each pair of methods, perform wilcoxon test
for i, method1 in enumerate(methods):
    for j, method2 in enumerate(methods):
        if i < j:
            t_stat, p_val = wilcoxon(external_all_df[method1], external_all_df[method2])
            significant = "(SIG)" if p_val < alpha else ""
            if p_val < alpha*100:
                print(f"{method1} vs {method2}: p-value = {p_val} {significant}")

            p_values_ext[i, j] = p_val

# turn to triangular matrix
p_val_ext_tri = np.triu(p_values_ext)
print(p_values_ext)
df_ext_p_values = pd.DataFrame(p_values_ext, index=methods, columns=methods)

In [ ]:
# convert p-values using bonferroni correction
df_ext_p_values_bonf = df_ext_p_values.applymap(lambda x: min(x * n_comparisons, 1))

In [ ]:
def format_p_value(val):
    return "<0.001" if val < 0.001 else f"{val:.3f}"

In [ ]:
# Trim empty first column and bottom row
df_ext_p_values_trimmed = df_ext_p_values.iloc[:-1, 1:]

# print shape of trimmed dataframe
print(df_ext_p_values_trimmed.shape)

annot_ext_pvals = np.vectorize(format_p_value)(df_ext_p_values_trimmed.values)

print(annot_ext_pvals.shape)

# format seaborn style
sns.set_style("ticks")

# set cmap as a gradient from white to blue 
#inverted cmap
cmap = colormaps["Blues_r"]

# make heatmap of p-values
plt.figure(figsize=(10, 6))
ax = sns.heatmap(df_ext_p_values_trimmed, annot=annot_ext_pvals, cmap=cmap, norm=norm, fmt="", mask=df_ext_p_values_trimmed==0, vmin=0, vmax=1, annot_kws={"size": 16})

# Set custom colorbar ticks and labels
cbar = ax.collections[0].colorbar
cbar.set_ticks(bounds)  # Explicitly include 0.0024
cbar.set_ticklabels(tick_labels)  # Force 0.0024 as a label
cbar.ax.tick_params(labelsize=16)

# move x ticks to top
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')

# Adjust tick labels
plt.xticks(fontsize=16, rotation=30)
plt.yticks(fontsize=16, rotation=0)
plt.tight_layout()
plt.savefig('p_values_external_dice_scores_Blues.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Repeat pairwise wilcoxon but with different correction
# use benjamini-hochberg correction for multiple comparisons
methods = external_all_dice_medians.index.tolist()

# get all p-values
p_values = []
comparisons = []
for i, method1 in enumerate(methods):
    for j, method2 in enumerate(methods):
        if i < j:
            t_stat, p_val = wilcoxon(external_all_df[method1], external_all_df[method2])
            p_values.append(p_val)
            comparisons.append((method1, method2))
# apply benjamini-hochberg correction
corrected_pvals = multipletests(p_values, method='fdr_bh')[1]
# print significant results
for i, (method1, method2) in enumerate(comparisons):
    if corrected_pvals[i] < 0.05:
        print(f"{method1} vs {method2}: p-value = {corrected_pvals[i]:.7f} (BH corrected)")

In [ ]:
# Can see that Zscore, CLAHE, and Nyúl are not significantly different
# from each other, but are significantly different from HE, GMM, min-max and robust min-max

In [ ]:
# bland-altman plot for zscore and zscore hist match
sm.graphics.mean_diff_plot(external_all_df["Zscore"], external_all_df["Nyul"])
plt.show()

In [ ]:
# Plot differences between two methods to check if they are normally distributed
# and if we can use t-test or wilcoxon test
diff = external_all_df["Zscore"] - external_all_df["Nyul"]

plt.figure(figsize=(10, 6))
sns.histplot(diff, bins=50)
plt.xlabel("Difference in Dice score")
plt.ylabel("Frequency")
plt.show()

# perform shapiro-wilk test to check for normality
from scipy.stats import shapiro

stat, p_val = shapiro(diff)
print(f"p-value = {p_val:.4f}")
print(stat)

# plot q-q plot
import scipy.stats as stats

plt.figure(figsize=(6, 6))
stats.probplot(diff, plot=plt)
plt.show()

## Now do hausdorff

In [ ]:
# get hausdorff distances from all external folders
external_all_hd_df = pd.DataFrame()

for folder in subfolders:
    json_path = os.path.join(data_path, folder, 'summary.json')
    hausdorff_distances = get_hausdorff_distances(json_path)
    external_all_hd_df[folder] = hausdorff_distances

external_all_hd_df.head()

In [ ]:
# rename columns
external_all_hd_df.columns = [
    "Z-score",
    "Min-max",
    "R. min-max",
    "HE",
    "CLAHE",
    "Nyúl",
    "GMM"
]

In [ ]:
external_all_hd_df.describe()

In [ ]:
# plot box plot of hausdorff distances
plt.figure(figsize=(10, 6))
sns.boxplot(data=external_all_hd_df)
plt.ylabel("Hausdorff distance")
plt.savefig('boxplot_external_all_hausdorff_distances.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# check if there is a significant difference between the methods
# Friedman test
# reorder data into correct format
data = external_all_hd_df.T.values

# perform friedman test
stat, p = friedmanchisquare(*data)
print(f"Friedman test statistic: {stat}, p-value: {p}")

# if p < 0.05, perform post-hoc tests
if p < 0.05:
    print("Friedman test is significant, can post-hoc tests")

In [ ]:
# get medians
external_all_hd_medians = external_all_hd_df.median()
# order columns by increasing median
external_all_hd_medians = external_all_hd_medians.sort_values(ascending=True)
# print medians
print("Medians:\n",external_all_hd_medians)

In [ ]:
# check if there is a significant difference between the methods
# perform wilcoxon test between all pairs of methods
# Use Bonferroni correction for multiple comparisons
methods = external_all_hd_medians.index.tolist()
# number of comparisons
n_comparisons = len(methods) * (len(methods) - 1) / 2
alpha = 0.05 / n_comparisons
print(f"Bonferroni corrected alpha: {alpha}")

# for each pair of methods, perform wilcoxon test
for i, method1 in enumerate(methods):
    for j, method2 in enumerate(methods):
        if i < j:
            t_stat, p_val = wilcoxon(external_all_hd_df[method1], external_all_hd_df[method2])
            significant = "(SIG)" if p_val < alpha else ""
            if p_val < alpha:
                print(f"{method1} vs {method2}: p-value = {p_val:.7f} {significant}")

In [ ]:
# get index of 2nd highest zscore hausdorff distance
external_all_hd_df.sort_values("Zscore", ascending=False).head(10)

In [ ]:
worst_im = 121

In [ ]:
# plot this image, along with the mask and prediction
# load image
skmtea_all_im_path = os.path.join(data_path, '../images_all_skmtea')
image_paths = [f.path for f in os.scandir(skmtea_all_im_path) if f.is_file()]
image_paths.sort()

# load image
image = sitk.ReadImage(image_paths[worst_im])
image_array = sitk.GetArrayFromImage(image)

# load mask
skmtea_all_mask_path = os.path.join(data_path, '../labels_all_skmtea')
mask_paths = [f.path for f in os.scandir(skmtea_all_mask_path) if f.is_file()]
mask_paths.sort()

mask = sitk.ReadImage(mask_paths[worst_im])
mask_array = sitk.GetArrayFromImage(mask)

# load prediction
zscore_all_preds_path = os.path.join(data_path, 'zscore_hist_match_preds_all_skmtea')
pred_paths = [f.path for f in os.scandir(zscore_all_preds_path) if f.is_file()]
pred_paths.sort()

pred = sitk.ReadImage(pred_paths[worst_im])
pred_array = sitk.GetArrayFromImage(pred)

In [ ]:
# now plot the image, mask, and prediction
plt.figure(figsize=(18, 10))
plt.subplot(1, 3, 1)
plt.imshow(image_array[100,...], cmap="gray")
plt.title("Image")
plt.axis("off")
plt.subplot(1, 3, 2)
plt.imshow(mask_array[100,...], cmap="gray")
plt.title("Mask")
plt.axis("off")
plt.subplot(1, 3, 3)
plt.imshow(pred_array[100,...], cmap="gray")
plt.title("Prediction")
plt.axis("off")
plt.show()

In [ ]:
# show mask and prediction, summed through axis 1
plt.figure(figsize=(18, 10))
plt.subplot(1, 2, 1)
plt.imshow(np.sum(mask_array, axis=1), cmap="gray")
plt.title("Mask")
plt.subplot(1, 2, 2)
plt.imshow(np.sum(pred_array, axis=1), cmap="gray")
plt.title("Prediction")
plt.show()

In [ ]:
# look at slices 40-80, 5 slice at a time
for i in range(30, 80, 5):
    plt.figure(figsize=(18, 10))
    plt.subplot(1, 3, 1)
    plt.imshow(image_array[i,...], cmap="gray")
    plt.title("Image")
    plt.axis("off")
    plt.subplot(1, 3, 2)
    plt.imshow(mask_array[i,...], cmap="gray")
    plt.title("Mask")
    plt.axis("off")
    plt.subplot(1, 3, 3)
    plt.imshow(pred_array[i,...], cmap="gray")
    plt.title("Prediction")
    plt.axis("off")
    plt.show()